In [1]:
import pandas as pd
import re


In [10]:
import pandas as pd

# Load Twitter dataset
tweets = pd.read_csv('..//data/raw/twcs.csv')



ParserError: Error tokenizing data. C error: Expected 7 fields in line 1042377, saw 8


In [3]:
print(tweets.shape)
print(tweets.head())


(2811774, 7)
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3                

In [4]:
# Load CRM ticket dataset
tickets = pd.read_csv('/kaggle/input/datasets/ajverse/customer-support-tickets-crm-dataset/customer_support_tickets.csv')


In [5]:
print(tickets.shape)
print(tickets.head())


(20000, 12)
    Ticket_ID     Customer_Name              Customer_Email  \
0  TKT-100000      George Simon  lisastrickland@example.com   
1  TKT-100001    Scott Thompson          wevans@example.org   
2  TKT-100002    Jennifer Smith        oleonard@example.net   
3  TKT-100003    Rachel Bullock     katherine67@example.net   
4  TKT-100004  Thomas Parks DDS       raykelsey@example.com   

                    Ticket_Subject  \
0  Hours of operation - Individual   
1          Data not syncing - Card   
2            2FA issues - Question   
3               Login failed - Let   
4        Refund status - Attention   

                                  Ticket_Description   Issue_Category  \
0  Hi Support, Where is your headquarters located...  General Inquiry   
1  Hi Support, The application crashes every time...        Technical   
2  Hi Support, How do I upgrade to the Enterprise...          Account   
3  Hi Support, The dashboard is not loading any d...        Technical   
4  Hi Support, 

In [6]:
# Twitter dataset
tweets_clean = tweets[['tweet_id', 'text', 'inbound']]
tweets_clean = tweets_clean.rename(columns={
    'tweet_id': 'id',
    'text': 'text',
    'inbound': 'source'
})
tweets_clean['category'] = 'twitter'

# CRM ticket dataset
tickets_clean = tickets[['Ticket_ID', 'Ticket_Description', 'Issue_Category']]
tickets_clean = tickets_clean.rename(columns={
    'Ticket_ID': 'id',
    'Ticket_Description': 'text',
    'Issue_Category': 'category'
})
tickets_clean['source'] = 'ticket'


In [7]:
merged_data = pd.concat([tweets_clean, tickets_clean], ignore_index=True)


In [8]:
# Show first 10 rows of the merged dataset
merged_data.head(10)


,id,text,source,category
0,1,@115712 I understand. I would like to assist y...,False,twitter
1,2,@sprintcare and how do you propose we do that,True,twitter
2,3,@sprintcare I have sent several private messag...,True,twitter
3,4,@115712 Please send us a Private Message so th...,False,twitter
4,5,@sprintcare I did.,True,twitter
5,6,@115712 Can you please send us a private messa...,False,twitter
6,8,@sprintcare is the worst customer service,True,twitter
7,11,@115713 This is saddening to hear. Please shoo...,False,twitter
8,12,@sprintcare You gonna magically change your co...,True,twitter
9,15,@115713 We understand your concerns and we'd l...,False,twitter


In [9]:
# Show first 5 CRM tickets + first 5 tweets
merged_data[merged_data['source']=='ticket'].head(5), merged_data[merged_data['source']=='twitter'].head(5)


(                 id                                               text  \
 2811774  TKT-100000  Hi Support, Where is your headquarters located...   
 2811775  TKT-100001  Hi Support, The application crashes every time...   
 2811776  TKT-100002  Hi Support, How do I upgrade to the Enterprise...   
 2811777  TKT-100003  Hi Support, The dashboard is not loading any d...   
 2811778  TKT-100004  Hi Support, I have been trying to update my pa...   
 
          source         category  
 2811774  ticket  General Inquiry  
 2811775  ticket        Technical  
 2811776  ticket          Account  
 2811777  ticket        Technical  
 2811778  ticket          Billing  ,
 Empty DataFrame
 Columns: [id, text, source, category]
 Index: [])

In [10]:
merged_data['source'].value_counts()


source
True      1537843
False     1273931
ticket      20000
Name: count, dtype: int64

In [11]:
merged_data.to_csv('/kaggle/working/final_merged_dataset.csv', index=False)


In [ ]:
import pandas as pd

# Load CRM ticket dataset
tickets = pd.read_csv('..//data/raw/tickets.csv', on_bad_lines='skip')

# Load Twitter dataset
tweets = pd.read_csv('..//data/raw/twcs.csv', on_bad_lines='skip')

# -----------------------------
# Standardize CRM tickets
# Keep only the relevant columns and rename them
tickets = tickets[['Ticket_ID', 'Ticket_Description', 'Issue_Category']]
tickets.rename(columns={
    'Ticket_ID': 'id',
    'Ticket_Description': 'text',
    'Issue_Category': 'category'
}, inplace=True)
tickets['source'] = 'ticket'

# -----------------------------
# Standardize Twitter dataset
# Keep only relevant columns and rename them
tweets = tweets[['tweet_id', 'text']]
tweets.rename(columns={
    'tweet_id': 'id'
}, inplace=True)
tweets['category'] = 'Customer Inquiry'  # Optional: generic category
tweets['source'] = 'twitter'

# -----------------------------
# Merge datasets
merged_data = pd.concat([tickets, tweets], ignore_index=True)

# -----------------------------
# Preview merged dataset
print(merged_data.head(10))
print("\nCounts per source:")
print(merged_data['source'].value_counts())
merged_data.to_csv('..//data/processed/merged_support_data.csv', index=False)

# Number of entries per category
print(merged_data['category'].value_counts())

# Sample entries from each source
print(merged_data[merged_data['source']=='twitter'].head(5))
print(merged_data[merged_data['source']=='ticket'].head(5))


           id                                               text  \
0  TKT-100000  Hi Support, Where is your headquarters located...   
1  TKT-100001  Hi Support, The application crashes every time...   
2  TKT-100002  Hi Support, How do I upgrade to the Enterprise...   
3  TKT-100003  Hi Support, The dashboard is not loading any d...   
4  TKT-100004  Hi Support, I have been trying to update my pa...   
5  TKT-100005  Hi Support, Where is your headquarters located...   
6  TKT-100006  Hi Support, How do I upgrade to the Enterprise...   
7  TKT-100007  Hi Support, My subscription renewed automatica...   
8  TKT-100008  Hi Support, I cannot log into my account even ...   
9  TKT-100009  Hi Support, Is there a roadmap for new feature...   

          category  source  
0  General Inquiry  ticket  
1        Technical  ticket  
2          Account  ticket  
3        Technical  ticket  
4          Billing  ticket  
5  General Inquiry  ticket  
6          Account  ticket  
7          Billing 

In [13]:
merged_data.to_csv('..//data/processed/merged_support_data.csv', index=False)


In [15]:
# Number of entries per category
print(merged_data['category'].value_counts())

# Sample entries from each source
print(merged_data[merged_data['source']=='twitter'].head(5))
print(merged_data[merged_data['source']=='ticket'].head(5))


category
Customer Inquiry    2811631
Technical              5918
Billing                5036
Account                4081
General Inquiry        3925
Fraud                  1040
Name: count, dtype: int64
      id                                               text          category  \
20000  1  @115712 I understand. I would like to assist y...  Customer Inquiry   
20001  2      @sprintcare and how do you propose we do that  Customer Inquiry   
20002  3  @sprintcare I have sent several private messag...  Customer Inquiry   
20003  4  @115712 Please send us a Private Message so th...  Customer Inquiry   
20004  5                                 @sprintcare I did.  Customer Inquiry   

        source  
20000  twitter  
20001  twitter  
20002  twitter  
20003  twitter  
20004  twitter  
           id                                               text  \
0  TKT-100000  Hi Support, Where is your headquarters located...   
1  TKT-100001  Hi Support, The application crashes every time...   
2  T